<a href="https://colab.research.google.com/github/revadirevathi20-sys/hdb-resale-etl-pipeline/blob/main/notebooks/hdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


HDB Resale ETL Pipeline - Full Notebook

This notebook runs the full ETL pipeline and displays all required outputs for verification against the assignment requirements

Requirements covered:
Data Quality  : Combine, Profile, Validate, Remaining Lease, Dedup, Anomalies
Transformation: Resale Identifier, Dedup, Hashing
Output        : Raw, Cleaned, Transformed, Failed, Hashed


In [1]:
# Clone the repository and set working directory

import os

repo_name = "hdb-resale-etl-pipeline"

if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/revadirevathi20-sys/hdb-resale-etl-pipeline.git

os.chdir(f"/content/{repo_name}")
print("✓ Working directory:", os.getcwd())

Cloning into 'hdb-resale-etl-pipeline'...
remote: Enumerating objects: 272, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 272 (delta 65), reused 15 (delta 15), pack-reused 159 (from 1)
Receiving objects: 100% (272/272), 10.92 MiB | 17.36 MiB/s, done.
Resolving deltas: 100% (135/135), done.
✓ Working directory: /content/hdb-resale-etl-pipeline


In [2]:
# Import Dependencies

import pandas as pd
import numpy as np
import hashlib
import warnings
warnings.filterwarnings("ignore")

print("✓ Dependencies loaded")

✓ Dependencies loaded


In [3]:
# Run Ingestion (Data Quality Req 1 — Combine Datasets)
# Combines all raw CSV files into a single master dataset filtered to Jan 2012 to Dec 2016
# Saves to data/cleaned/master_raw_combined.csv.

import subprocess, sys

print("=" * 60)
print("STEP 1: INGESTION — Combining raw datasets")
print("=" * 60)

result = subprocess.run(
    [sys.executable, "src/ingest.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

STEP 1: INGESTION — Combining raw datasets
/content/hdb-resale-etl-pipeline/src/../data/raw/ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv
True
[ResaleFlatPricesBasedonApprovalDate2000Feb2012.csv] Filtered out 22281 rows before 2012-01
/content/hdb-resale-etl-pipeline/src/../data/raw/ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv
True
/content/hdb-resale-etl-pipeline/src/../data/raw/ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv
True
/content/hdb-resale-etl-pipeline/src/../data/raw/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv
True

Master dataset shape: (92544, 11)
Date range: 2012-01-01 00:00:00 to 2016-12-01 00:00:00
Master dataset saved to ../data/combined/master_raw_combined.csv



In [5]:
# Verify Raw Output
# Data Output Req — RAW: Displays the combined master dataset as-is.

print("=" * 60)
print("OUTPUT — RAW: master_raw_combined.csv")
print("=" * 60)

raw = pd.read_csv("../data/combined/master_raw_combined.csv", low_memory=False)

print(f"Shape       : {raw.shape[0]:,} rows × {raw.shape[1]} columns")
print(f"Date range  : {raw['month'].min()} → {raw['month'].max()}")
print(f"Columns     : {list(raw.columns)}")
print()
print("── First 5 rows ──")
display(raw.head())

print()
print("── Last 5 rows ──")
display(raw.tail())

OUTPUT — RAW: master_raw_combined.csv
Shape       : 92,544 rows × 11 columns
Date range  : 2012-01-01 → 2016-12-01
Columns     : ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease']

── First 5 rows ──


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
0,2012-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,257800.0,NaN
1,2012-01-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,263000.0,NaN
2,2012-01-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,275000.0,NaN
3,2012-01-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,260000.0,NaN
4,2012-01-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,226000.0,NaN



── Last 5 rows ──


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
92539,2016-12-01,YISHUN,5 ROOM,297,YISHUN ST 20,13 TO 15,112.0,Improved,2000,488000.0,82.0
92540,2016-12-01,YISHUN,5 ROOM,838,YISHUN ST 81,01 TO 03,122.0,Improved,1987,455000.0,69.0
92541,2016-12-01,YISHUN,EXECUTIVE,664,YISHUN AVE 4,10 TO 12,181.0,Apartment,1992,778000.0,74.0
92542,2016-12-01,YISHUN,EXECUTIVE,325,YISHUN CTRL,01 TO 03,146.0,Maisonette,1988,575000.0,70.0
92543,2016-12-01,YISHUN,MULTI-GENERATION,666,YISHUN AVE 4,10 TO 12,164.0,Multi Generation,1987,735000.0,70.0


In [6]:
# Run Cleaning (Data Quality Req 2–7)
# Performs:
#   Req 2 — Data profiling
#   Req 3 — Validation rules (date, town, flat type, flat model, storey_range)
#   Req 4 — Recompute remaining lease (99-year HDB lease, rounded down)
#   Req 5 — Duplicate key handling (keep higher resale price)
#   Req 6 — Anomalous price detection (z-score heuristic)
#   Req 7 — Additional cleaning (whitespace, casing, floor area bounds)

print("=" * 60)
print("STEP 2: CLEANING — Data Quality Requirements 2–7")
print("=" * 60)

result = subprocess.run(
    [sys.executable, "src/clean.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

STEP 2: CLEANING — Data Quality Requirements 2–7

STDERR: Traceback (most recent call last):
  File "/content/hdb-resale-etl-pipeline/src/clean.py", line 225, in <module>
    df = pd.read_csv(
         ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1880, in _make_engine
    self.handles = get_handle(
                   ^^^^^^^^^^^
  File "/us

In [ ]:
# Verify Cleaned Output
# Data Output Req — CLEANED: Dataset that passes all data quality requirements.

print("=" * 60)
print("OUTPUT — CLEANED: hdb_resale_cleaned.csv")
print("=" * 60)

cleaned = pd.read_csv("../data/cleaned/hdb_resale_cleaned.csv", low_memory=False)

print(f"Shape             : {cleaned.shape[0]:,} rows × {cleaned.shape[1]} columns")
print(f"Null counts       :\n{cleaned.isnull().sum()}")
print()
print("── Sample rows ──")
display(cleaned.head(10))

print()
print("── Remaining Lease column (Data Quality Req 4) ──")
display(cleaned[["month", "lease_commence_date", "remaining_lease", "lease_end_year"]].head(10))

print()
print("── Unique towns in cleaned data ──")
print(sorted(cleaned["town"].unique()))

print()
print("── Unique flat types in cleaned data ──")
print(sorted(cleaned["flat_type"].unique()))

print()
print("── Resale price statistics ──")
display(cleaned["resale_price"].describe())

OUTPUT — CLEANED: hdb_resale_cleaned.csv
Shape             : 90,475 rows × 12 columns
Null counts       :
month                  0
town                   0
flat_type              0
block                  0
street_name            0
storey_range           0
floor_area_sqm         0
flat_model             0
lease_commence_date    0
resale_price           0
remaining_lease        0
lease_end_year         0
dtype: int64

── Sample rows ──


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,lease_end_year
0,2016-09-01,CENTRAL AREA,5 ROOM,1G,CANTONMENT RD,43 TO 45,106.00,TYPE S2,2011,"1,120,000.00",93 YEARS 4 MONTHS,2110
1,2016-11-01,CENTRAL AREA,5 ROOM,1D,CANTONMENT RD,46 TO 48,107.00,TYPE S2,2011,"1,100,000.00",93 YEARS 2 MONTHS,2110
2,2016-11-01,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,31 TO 33,106.00,TYPE S2,2011,"1,100,000.00",93 YEARS 2 MONTHS,2110
3,2015-11-01,CENTRAL AREA,5 ROOM,1D,CANTONMENT RD,43 TO 45,107.00,TYPE S2,2011,"1,088,000.00",94 YEARS 2 MONTHS,2110
4,2016-06-01,CENTRAL AREA,5 ROOM,1D,CANTONMENT RD,34 TO 36,107.00,TYPE S2,2011,"1,070,000.00",93 YEARS 7 MONTHS,2110
5,2016-08-01,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,31 TO 33,105.00,TYPE S2,2011,"1,070,000.00",93 YEARS 5 MONTHS,2110
6,2016-01-01,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,28 TO 30,108.00,TYPE S2,2011,"1,068,888.00",94 YEARS 0 MONTHS,2110
7,2016-05-01,CENTRAL AREA,5 ROOM,1D,CANTONMENT RD,19 TO 21,106.00,TYPE S2,2011,"1,063,888.00",93 YEARS 8 MONTHS,2110
8,2015-04-01,CENTRAL AREA,5 ROOM,1G,CANTONMENT RD,28 TO 30,107.00,TYPE S2,2011,"1,060,000.00",94 YEARS 9 MONTHS,2110
9,2016-09-01,CENTRAL AREA,5 ROOM,1C,CANTONMENT RD,40 TO 42,106.00,TYPE S2,2011,"1,055,000.00",93 YEARS 4 MONTHS,2110



── Remaining Lease column (Data Quality Req 4) ──


,month,lease_commence_date,remaining_lease,lease_end_year
0,2016-09-01,2011,93 YEARS 4 MONTHS,2110
1,2016-11-01,2011,93 YEARS 2 MONTHS,2110
2,2016-11-01,2011,93 YEARS 2 MONTHS,2110
3,2015-11-01,2011,94 YEARS 2 MONTHS,2110
4,2016-06-01,2011,93 YEARS 7 MONTHS,2110
5,2016-08-01,2011,93 YEARS 5 MONTHS,2110
6,2016-01-01,2011,94 YEARS 0 MONTHS,2110
7,2016-05-01,2011,93 YEARS 8 MONTHS,2110
8,2015-04-01,2011,94 YEARS 9 MONTHS,2110
9,2016-09-01,2011,93 YEARS 4 MONTHS,2110



── Unique towns in cleaned data ──
['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT BATOK', 'BUKIT MERAH', 'BUKIT PANJANG', 'BUKIT TIMAH', 'CENTRAL AREA', 'CHOA CHU KANG', 'CLEMENTI', 'GEYLANG', 'HOUGANG', 'JURONG EAST', 'JURONG WEST', 'KALLANG/WHAMPOA', 'MARINE PARADE', 'PASIR RIS', 'PUNGGOL', 'QUEENSTOWN', 'SEMBAWANG', 'SENGKANG', 'SERANGOON', 'TAMPINES', 'TOA PAYOH', 'WOODLANDS', 'YISHUN']

── Unique flat types in cleaned data ──
['1 ROOM', '2 ROOM', '3 ROOM', '4 ROOM', '5 ROOM', 'EXECUTIVE', 'MULTI-GENERATION']

── Resale price statistics ──


,resale_price
count,"90,475.00"
mean,"450,771.89"
std,"127,356.30"
min,"192,000.00"
25%,"358,000.00"
50%,"428,000.00"
75%,"515,000.00"
max,"1,120,000.00"


In [ ]:
# Verify Failed Outputs
# Data Output Req — FAILED: Records removed at each cleaning stage.

print("=" * 60)
print("OUTPUT — FAILED: All failed record groups")
print("=" * 60)

failed_files = {
    "Validation failures"          : "../data/failed/failed_validation.csv",
    "Duplicate key failures"       : "../data/failed/failed_duplicates.csv",
    "Anomalous price failures"     : "../data/failed/failed_anomalies.csv",
    "Additional cleaning failures" : "../data/failed/failed_additional_cleaning.csv",
}

for label, path in failed_files.items():
    if os.path.exists(path):
        df_fail = pd.read_csv(path, low_memory=False)
        print(f"\n── {label}: {df_fail.shape[0]:,} records ──")
        display(df_fail.head(5))
    else:
        print(f"\n── {label}: file not found at {path} ──")

OUTPUT — FAILED: All failed record groups

── Validation failures: 0 records ──


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,fail_reason



── Duplicate key failures: 1,600 records ──


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,lease_end_year,fail_reason
0,2013-07-01,BUKIT TIMAH,EXECUTIVE,7,TOH YI DR,04 TO 06,146.00,Maisonette,1989,"950,000.00",74 Years 6 Months,2088,duplicate_key_lower_price
1,2015-11-01,CENTRAL AREA,4 ROOM,1A,CANTONMENT RD,37 TO 39,95.00,Type S1,2011,"915,000.00",94 Years 2 Months,2110,duplicate_key_lower_price
2,2013-07-01,TOA PAYOH,5 ROOM,81,LOR 4 TOA PAYOH,19 TO 21,122.00,Improved,1997,"880,000.00",82 Years 6 Months,2096,duplicate_key_lower_price
3,2016-10-01,BISHAN,5 ROOM,273B,BISHAN ST 24,01 TO 03,120.00,DBSS,2011,"878,888.00",93 Years 3 Months,2110,duplicate_key_lower_price
4,2013-07-01,BISHAN,EXECUTIVE,187,BISHAN ST 13,07 TO 09,146.00,Maisonette,1987,"878,000.00",72 Years 6 Months,2086,duplicate_key_lower_price



── Anomalous price failures: 469 records ──


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,lease_end_year,fail_reason
0,2016-12-01,KALLANG/WHAMPOA,3 ROOM,57,JLN MA'MOR,01 TO 03,259.00,Terrace,1972,"1,150,000.00",54 Years 1 Months,2071,anomalous_price
1,2016-08-01,KALLANG/WHAMPOA,5 ROOM,8,BOON KENG RD,28 TO 30,119.00,DBSS,2011,"1,100,000.00",93 Years 5 Months,2110,anomalous_price
2,2014-10-01,BISHAN,EXECUTIVE,194,BISHAN ST 13,22 TO 24,150.00,Maisonette,1987,"1,088,888.00",71 Years 3 Months,2086,anomalous_price
3,2015-03-01,KALLANG/WHAMPOA,3 ROOM,53,JLN MA'MOR,01 TO 03,280.00,Terrace,1972,"1,060,000.00",55 Years 10 Months,2071,anomalous_price
4,2016-10-01,BISHAN,5 ROOM,273B,BISHAN ST 24,40 TO 42,120.00,DBSS,2011,"1,038,000.00",93 Years 3 Months,2110,anomalous_price



── Additional cleaning failures: 0 records ──


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,lease_end_year,fail_reason


In [ ]:
# Cleaning Summary
# Quick reconciliation to confirm record counts add up correctly.

print("=" * 60)
print("CLEANING RECONCILIATION SUMMARY")
print("=" * 60)

raw_count       = pd.read_csv("../data/combined/master_raw_combined.csv").shape[0]
cleaned_count   = cleaned.shape[0]

failed_counts = {}
for label, path in failed_files.items():
    if os.path.exists(path):
        failed_counts[label] = pd.read_csv(path, low_memory=False).shape[0]
    else:
        failed_counts[label] = 0

total_failed = sum(failed_counts.values())

print(f"  Raw records              : {raw_count:>8,}")
for label, count in failed_counts.items():
    print(f"  {label:<30}: {count:>8,}")
print(f"  {'─'*42}")
print(f"  Final cleaned records    : {cleaned_count:>8,}")
print(f"  Check (raw - failed)     : {raw_count - total_failed:>8,}  ← should match cleaned")

CLEANING RECONCILIATION SUMMARY
  Raw records              :   92,544
  Validation failures           :        0
  Duplicate key failures        :    1,600
  Anomalous price failures      :      469
  Additional cleaning failures  :        0
  ──────────────────────────────────────────
  Final cleaned records    :   90,475
  Check (raw - failed)     :   90,475  ← should match cleaned


In [ ]:
# Run Transformation (Transformation Req 1–3)
#   Performs:
#   Req 1 — Create Resale Identifier (S + block digits + avg price digits +
#            month digits + town initial)
#   Req 2 — Deduplicate on identifier (keep higher resale price)
#   Req 3 — Hash identifier using SHA-256 (irreversible, collision-resistant)

print("=" * 60)
print("STEP 3: TRANSFORMATION — Transformation Requirements 1–3")
print("=" * 60)

result = subprocess.run(
    [sys.executable, "src/transform.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

STEP 3: TRANSFORMATION — Transformation Requirements 1–3

========== START TRANSFORMATION ==========

Transform duplicates: 13606 removed, 76869 kept

========== TRANSFORMATION SUMMARY ==========
Input records               : 90,475
Duplicate identifiers       : 13,606
Final transformed records   : 76,869

Files saved:
../data/transformed/hdb_resale_transformed.csv
../data/transformed/failed_transform_duplicates.csv



In [ ]:
# Verify Transformed Output
# Data Output Req — TRANSFORMED: Cleaned data with Resale Identifier added.
# Plain resale_identifier is retained here before hashing.

print("=" * 60)
print("OUTPUT — TRANSFORMED: hdb_resale_transformed.csv")
print("=" * 60)

transformed = pd.read_csv("../data/transformed/hdb_resale_transformed.csv", low_memory=False)

print(f"Shape   : {transformed.shape[0]:,} rows × {transformed.shape[1]} columns")
print(f"Columns : {list(transformed.columns)}")
print()
print("── Sample rows with Resale Identifier ──")
display(transformed[["month", "town", "flat_type", "block", "resale_price", "resale_identifier"]].head(10))

print()
print("── Identifier format check (should start with 'S') ──")
starts_with_s = transformed["resale_identifier"].str.startswith("S").all()
print(f"  All identifiers start with 'S': {starts_with_s}")

print()
print("── Identifier length distribution ──")
print(transformed["resale_identifier"].str.len().value_counts())

print()
print("── Uniqueness check ──")
total       = transformed.shape[0]
unique_ids  = transformed["resale_identifier"].nunique()
print(f"  Total records      : {total:,}")
print(f"  Unique identifiers : {unique_ids:,}")
print(f"  All unique         : {total == unique_ids}")

OUTPUT — TRANSFORMED: hdb_resale_transformed.csv
Shape   : 76,869 rows × 14 columns
Columns : ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease', 'lease_end_year', 'resale_identifier', 'resale_identifier_hashed']

── Sample rows with Resale Identifier ──


,month,town,flat_type,block,resale_price,resale_identifier
0,2012-01-01,ANG MO KIO,4 ROOM,102,"500,000.00",S1024701A
1,2012-01-01,ANG MO KIO,3 ROOM,118,"320,000.00",S1183401A
2,2012-01-01,ANG MO KIO,3 ROOM,121,"382,800.00",S1213401A
3,2012-01-01,ANG MO KIO,4 ROOM,126,"452,000.00",S1264701A
4,2012-01-01,ANG MO KIO,3 ROOM,151,"302,000.00",S1513401A
5,2012-01-01,ANG MO KIO,3 ROOM,154,"321,000.00",S1543401A
6,2012-01-01,ANG MO KIO,3 ROOM,157,"297,000.00",S1573401A
7,2012-01-01,ANG MO KIO,3 ROOM,163,"340,000.00",S1633401A
8,2012-01-01,ANG MO KIO,2 ROOM,170,"260,000.00",S1702501A
9,2012-01-01,ANG MO KIO,3 ROOM,174,"281,000.00",S1743401A



── Identifier format check (should start with 'S') ──
  All identifiers start with 'S': True

── Identifier length distribution ──
resale_identifier
9    76869
Name: count, dtype: int64

── Uniqueness check ──
  Total records      : 76,869
  Unique identifiers : 76,869
  All unique         : True


In [ ]:
# Verify Transform Failed Output
# Records removed during transformation deduplication (Transformation Req 2).

print("=" * 60)
print("OUTPUT — FAILED (Transform): failed_transform_duplicates.csv")
print("=" * 60)

transform_failed_path = "../data/failed/failed_transform_duplicates.csv"

if os.path.exists(transform_failed_path):
    tf_failed = pd.read_csv(transform_failed_path, low_memory=False)
    print(f"Shape   : {tf_failed.shape[0]:,} rows × {tf_failed.shape[1]} columns")
    print()
    display(tf_failed.head(10))
else:
    print("No transform duplicates found — file not created.")

OUTPUT — FAILED (Transform): failed_transform_duplicates.csv
Shape   : 13,606 rows × 14 columns



,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,lease_end_year,resale_identifier,fail_reason
0,2016-11-01,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,31 TO 33,106.00,TYPE S2,2011,"1,100,000.00",93 YEARS 2 MONTHS,2110,S0011011C,transform_duplicate_identifier
1,2016-09-01,CENTRAL AREA,5 ROOM,1C,CANTONMENT RD,40 TO 42,106.00,TYPE S2,2011,"1,055,000.00",93 YEARS 4 MONTHS,2110,S0011009C,transform_duplicate_identifier
2,2015-04-01,CENTRAL AREA,5 ROOM,1A,CANTONMENT RD,46 TO 48,107.00,TYPE S2,2011,"1,050,000.00",94 YEARS 9 MONTHS,2110,S0011004C,transform_duplicate_identifier
3,2015-02-01,CENTRAL AREA,5 ROOM,1A,CANTONMENT RD,37 TO 39,107.00,TYPE S2,2011,"1,030,000.00",94 YEARS 11 MONTHS,2110,S0019902C,transform_duplicate_identifier
4,2016-01-01,CENTRAL AREA,5 ROOM,1F,CANTONMENT RD,28 TO 30,107.00,TYPE S2,2011,"1,020,000.00",94 YEARS 0 MONTHS,2110,S0019401C,transform_duplicate_identifier
5,2015-02-01,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,22 TO 24,105.00,TYPE S2,2011,"1,000,000.00",94 YEARS 11 MONTHS,2110,S0019902C,transform_duplicate_identifier
6,2016-09-01,CENTRAL AREA,5 ROOM,1F,CANTONMENT RD,22 TO 24,105.00,TYPE S2,2011,"998,000.00",93 YEARS 4 MONTHS,2110,S0011009C,transform_duplicate_identifier
7,2016-09-01,CENTRAL AREA,5 ROOM,1D,CANTONMENT RD,31 TO 33,106.00,TYPE S2,2011,"995,000.00",93 YEARS 4 MONTHS,2110,S0011009C,transform_duplicate_identifier
8,2015-02-01,CENTRAL AREA,5 ROOM,1C,CANTONMENT RD,28 TO 30,107.00,TYPE S2,2011,"991,000.00",94 YEARS 11 MONTHS,2110,S0019902C,transform_duplicate_identifier
9,2015-05-01,CENTRAL AREA,5 ROOM,1A,CANTONMENT RD,31 TO 33,107.00,TYPE S2,2011,"980,000.00",94 YEARS 8 MONTHS,2110,S0019505C,transform_duplicate_identifier


In [ ]:
# Verify Hashed Output
# Data Output Req — HASHED: Cleaned data + hashed identifier column.
# Plain resale_identifier is dropped; only the SHA-256 hash is retained.
#
# Hashing algorithm: SHA-256
#   - Irreversible: cannot reconstruct original identifier from hash
#   - Deterministic: same input always produces same hash
#   - Collision-resistant: negligible probability of two different identifiers
#     producing the same hash (2^256 possible outputs)
#   - Uniqueness preserved: since identifiers are unique, their hashes are too

print("=" * 60)
print("OUTPUT — HASHED: hdb_resale_hashed.csv")
print("=" * 60)

hashed_path = "../data/hashed/hdb_resale_hashed.csv"

if os.path.exists(hashed_path):
    hashed = pd.read_csv(hashed_path, low_memory=False)

    print(f"Shape   : {hashed.shape[0]:,} rows × {hashed.shape[1]} columns")
    print(f"Columns : {list(hashed.columns)}")
    print()
    print("── Sample rows with hashed identifier ──")
    display(hashed[["month", "town", "flat_type", "resale_price", "resale_identifier_hashed"]].head(10))

    print()
    print("── Confirm plain resale_identifier is NOT present ──")
    plain_present = "resale_identifier" in hashed.columns
    print(f"  Plain identifier present: {plain_present}  ← should be False")

    print()
    print("── Hash format check (SHA-256 = 64 hex characters) ──")
    hash_lengths = hashed["resale_identifier_hashed"].str.len().value_counts()
    print(hash_lengths)
    all_64 = (hashed["resale_identifier_hashed"].str.len() == 64).all()
    print(f"  All hashes are 64 characters: {all_64}")

    print()
    print("── Hash uniqueness check ──")
    total_hashed  = hashed.shape[0]
    unique_hashes = hashed["resale_identifier_hashed"].nunique()
    print(f"  Total records      : {total_hashed:,}")
    print(f"  Unique hashes      : {unique_hashes:,}")
    print(f"  All unique         : {total_hashed == unique_hashes}")
else:
    print(f"Hashed file not found at {hashed_path}")

OUTPUT — HASHED: hdb_resale_hashed.csv
Hashed file not found at ../data/hashed/hdb_resale_hashed.csv


In [ ]:
# Full Pipeline Summary
# End-to-end record count reconciliation across all output files.

print("=" * 60)
print("FULL PIPELINE SUMMARY")
print("=" * 60)

summary = {
    "1. Raw (combined)"                  : "../data/combined/master_raw_combined.csv",
    "2. Cleaned"                         : "../data/cleaned/hdb_resale_cleaned.csv",
    "3. Transformed"                     : "../data/transformed/hdb_resale_transformed.csv",
    "4. Hashed"                          : "../data/hashed/hdb_resale_hashed.csv",
    "5. Failed — Validation"             : "../data/failed/failed_validation.csv",
    "6. Failed — Duplicates (clean)"     : "../data/failed/failed_duplicates.csv",
    "7. Failed — Anomalous prices"       : "../data/failed/failed_anomalies.csv",
    "8. Failed — Additional cleaning"    : "../data/failed/failed_additional_cleaning.csv",
    "9. Failed — Transform duplicates"   : "../data/failed/failed_transform_duplicates.csv",
}

for label, path in summary.items():
    if os.path.exists(path):
        count = pd.read_csv(path, low_memory=False).shape[0]
        print(f"  {label:<40}: {count:>8,} records")
    else:
        print(f"  {label:<40}: {'file not found':>15}")

print()
print("All output files verified. Pipeline complete.")

FULL PIPELINE SUMMARY
  1. Raw (combined)                       :   92,544 records
  2. Cleaned                              :   90,475 records
  3. Transformed                          :   76,869 records
  4. Hashed                               :  file not found
  5. Failed — Validation                  :        0 records
  6. Failed — Duplicates (clean)          :    1,600 records
  7. Failed — Anomalous prices            :      469 records
  8. Failed — Additional cleaning         :        0 records
  9. Failed — Transform duplicates        :   13,606 records

All output files verified. Pipeline complete.
